# 📤 Notebook 05 — Final Load Preparation for Tableau
### HR Analytics: Job Change of Data Scientists
**Objective:** Produce a clean, business-readable, Tableau-optimized CSV.
**Output:** `data/processed/hr_tableau_ready.csv`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

COLORS = {'primary': '#1B3A6B', 'accent': '#E84855', 'safe': '#2ECC71'}
plt.rcParams.update({'figure.figsize': (12, 5), 'axes.spines.top': False, 'axes.spines.right': False})

df = pd.read_csv('../data/processed/hr_cleaned.csv')
print(f'Input: {df.shape}')

## 1. Select & Rename Columns for Business Readability

In [ ]:
tableau = df[[
    'enrollee_id', 'city', 'city_development_index', 'city_tier',
    'gender', 'relevent_experience', 'has_relevent_exp',
    'enrolled_university', 'is_enrolled',
    'education_level', 'education_ordinal',
    'major_discipline', 'is_stem',
    'experience', 'experience_num', 'experience_band',
    'company_size', 'company_size_num',
    'company_type',
    'last_new_job', 'last_new_job_num',
    'training_hours', 'training_hours_capped',
    'retention_risk_score', 'risk_tier',
    'target'
]].copy()

# Human-readable rename map
rename_map = {
    'enrollee_id':              'Candidate_ID',
    'city':                     'City_Code',
    'city_development_index':   'City_Development_Index',
    'city_tier':                'City_Tier',
    'gender':                   'Gender',
    'relevent_experience':      'Relevant_Experience_Raw',
    'has_relevent_exp':         'Has_Relevant_Experience',
    'enrolled_university':      'University_Enrollment',
    'is_enrolled':              'Is_Enrolled_University',
    'education_level':          'Education_Level',
    'education_ordinal':        'Education_Score',
    'major_discipline':         'Major_Discipline',
    'is_stem':                  'Is_STEM',
    'experience':               'Experience_Raw',
    'experience_num':           'Experience_Years',
    'experience_band':          'Experience_Band',
    'company_size':             'Company_Size',
    'company_size_num':         'Company_Size_Score',
    'company_type':             'Company_Type',
    'last_new_job':             'Last_Job_Gap',
    'last_new_job_num':         'Last_Job_Gap_Years',
    'training_hours':           'Training_Hours_Raw',
    'training_hours_capped':    'Training_Hours',
    'retention_risk_score':     'Retention_Risk_Score',
    'risk_tier':                'Risk_Tier',
    'target':                   'Job_Change_Intent'
}
tableau = tableau.rename(columns=rename_map)

# Human-readable target label
tableau['Job_Change_Label'] = tableau['Job_Change_Intent'].map({0: 'Not Switching', 1: 'Switching'})

print(f'Tableau dataset shape: {tableau.shape}')
tableau.head()

## 2. Add Derived KPI Columns for Tableau Calculated Fields

In [ ]:
# Overall Job Change Rate (for reference / LOD calc in Tableau)
overall_jcr = tableau['Job_Change_Intent'].mean()
tableau['Overall_JCR_Pct'] = round(overall_jcr * 100, 2)

# Training Intensity (normalized 0–1)
t_min = tableau['Training_Hours'].min()
t_max = tableau['Training_Hours'].max()
tableau['Training_Intensity'] = ((tableau['Training_Hours'] - t_min) / (t_max - t_min)).round(4)

# CDI Band for Tableau grouping
tableau['CDI_Band'] = pd.cut(
    tableau['City_Development_Index'],
    bins=[0, 0.55, 0.65, 0.75, 0.85, 1.0],
    labels=['<0.55', '0.55–0.65', '0.65–0.75', '0.75–0.85', '>0.85']
).astype(str)

# Binary risk flag
tableau['Is_High_Risk'] = (tableau['Risk_Tier'] == 'High Risk').astype(int)

print('Derived KPI columns added:')
print(['Overall_JCR_Pct', 'Training_Intensity', 'CDI_Band', 'Is_High_Risk'])
tableau[['Candidate_ID', 'Retention_Risk_Score', 'Risk_Tier', 'Is_High_Risk',
         'Training_Intensity', 'CDI_Band', 'Job_Change_Label']].head(10)

## 3. Final Quality Checks

In [ ]:
print('──── FINAL DATA QUALITY CHECK ────────────────────────────')
print(f'  Total rows          : {len(tableau):,}')
print(f'  Total columns       : {len(tableau.columns)}')
print(f'  Null values         : {tableau.isnull().sum().sum()}')
print(f'  Duplicate IDs       : {tableau.duplicated(subset="Candidate_ID").sum()}')
print(f'  Job Change Rate     : {tableau["Job_Change_Intent"].mean()*100:.2f}%')
print(f'  High Risk Candidates: {tableau["Is_High_Risk"].sum():,} ({tableau["Is_High_Risk"].mean()*100:.1f}%)')
print(f'  Avg Risk Score      : {tableau["Retention_Risk_Score"].mean():.2f}/10')
print(f'  Avg Training Hours  : {tableau["Training_Hours"].mean():.1f}')
print('──────────────────────────────────────────────────────────')
print()
print('Risk Tier Distribution:')
print(tableau['Risk_Tier'].value_counts().to_string())
print()
print('City Tier Distribution:')
print(tableau['City_Tier'].value_counts().to_string())

## 4. KPI Summary Dashboard (Pre-Tableau Validation)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(20, 10))
axes = axes.flatten()

# KPI 1: JCR by City Tier
ct_jcr = tableau.groupby('City_Tier')['Job_Change_Intent'].mean().mul(100)
axes[0].bar(ct_jcr.index, ct_jcr.values, color=[COLORS['accent'], COLORS['warn'], COLORS['safe']], edgecolor='white')
for i, v in enumerate(ct_jcr.values): axes[0].text(i, v+0.4, f'{v:.1f}%', ha='center', fontweight='bold')
axes[0].set_title('KPI 1: JCR by City Tier'); axes[0].set_ylabel('Job Change Rate %')

# KPI 2: JCR by Experience Band
exp_order = ['Fresher (<1yr)', 'Junior (1-3yr)', 'Mid (4-7yr)', 'Senior (8-15yr)', 'Veteran (>15yr)']
eb_jcr = tableau.groupby('Experience_Band')['Job_Change_Intent'].mean().mul(100).reindex(exp_order)
axes[1].bar(range(len(eb_jcr)), eb_jcr.values, color=COLORS['primary'], alpha=0.85, edgecolor='white')
axes[1].set_xticks(range(len(eb_jcr)))
axes[1].set_xticklabels(eb_jcr.index, rotation=25, fontsize=8)
for i, v in enumerate(eb_jcr.values): axes[1].text(i, v+0.3, f'{v:.1f}%', ha='center', fontsize=8)
axes[1].set_title('KPI 2: JCR by Experience Band'); axes[1].set_ylabel('Job Change Rate %')

# KPI 3: Risk Tier Distribution
rt_counts = tableau['Risk_Tier'].value_counts().reindex(['Low Risk', 'Medium Risk', 'High Risk'])
axes[2].pie(rt_counts.values, labels=rt_counts.index,
            colors=[COLORS['safe'], COLORS['warn'], COLORS['accent']],
            autopct='%1.1f%%', startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[2].set_title('KPI 3: Risk Tier Distribution')

# KPI 4: Risk Score vs JCR
rr_bins = pd.cut(tableau['Retention_Risk_Score'], bins=8)
rr_jcr = tableau.groupby(rr_bins)['Job_Change_Intent'].mean().mul(100)
axes[3].plot(range(len(rr_jcr)), rr_jcr.values, color=COLORS['accent'], lw=2.5, marker='o')
axes[3].set_title('KPI 4: Retention Risk Score vs JCR')
axes[3].set_ylabel('Job Change Rate %'); axes[3].set_xlabel('Risk Score Bin')

# KPI 5: Company Type JCR
comp_jcr = tableau.groupby('Company_Type')['Job_Change_Intent'].mean().mul(100).sort_values()
axes[4].barh(comp_jcr.index, comp_jcr.values, color=COLORS['primary'], alpha=0.85)
for i, v in enumerate(comp_jcr.values): axes[4].text(v+0.2, i, f'{v:.1f}%', va='center', fontsize=8)
axes[4].set_title('KPI 5: JCR by Company Type'); axes[4].set_xlabel('Job Change Rate %')

# KPI 6: Education Level JCR
edu_order = ['Primary School', 'High School', 'Graduate', 'Masters', 'Phd']
edu_jcr = tableau.groupby('Education_Level')['Job_Change_Intent'].mean().mul(100).reindex(edu_order)
axes[5].bar(range(len(edu_jcr)), edu_jcr.values, color=COLORS['primary'], alpha=0.85, edgecolor='white')
axes[5].set_xticks(range(len(edu_jcr)))
axes[5].set_xticklabels(edu_jcr.index, rotation=20, fontsize=8)
for i, v in enumerate(edu_jcr.values):
    if not np.isnan(v): axes[5].text(i, v+0.3, f'{v:.1f}%', ha='center', fontsize=8)
axes[5].set_title('KPI 6: JCR by Education Level'); axes[5].set_ylabel('Job Change Rate %')

plt.suptitle('Pre-Tableau KPI Validation Dashboard', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/processed/05_kpi_validation_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Export Tableau-Ready CSV

In [ ]:
TABLEAU_PATH = '../data/processed/hr_tableau_ready.csv'
tableau.to_csv(TABLEAU_PATH, index=False)

print('='*60)
print('  TABLEAU EXPORT COMPLETE')
print('='*60)
print(f'  File : {TABLEAU_PATH}')
print(f'  Rows : {len(tableau):,}')
print(f'  Cols : {len(tableau.columns)}')
print(f'  Size : {tableau.memory_usage(deep=True).sum()/1e6:.2f} MB')
print()
print('  Column List:')
for col in tableau.columns:
    print(f'    • {col}')
print('='*60)
print()
print('📌 NEXT STEP: Import hr_tableau_ready.csv into Tableau Desktop')
print('   → Connect to Text File → hr_tableau_ready.csv')
print('   → Build dashboards per tableau/dashboard_links.md wireframe')

---
## ✅ All 5 Notebooks Complete!

| Notebook | Status | Output |
|----------|--------|--------|
| 01_data_sourcing | ✅ | Audit report + visualizations |
| 02_cleaning | ✅ | `hr_cleaned.csv` + `hr_engineered.csv` |
| 03_eda | ✅ | 6 analytical charts |
| 04_statistical_analysis | ✅ | Hypothesis tests + models |
| 05_final_load_prep | ✅ | `hr_tableau_ready.csv` |

**→ Proceed to Tableau dashboard building per `tableau/dashboard_links.md`**